In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

# -------------------- 路径与设置 --------------------

RESULTS_DIR = Path(
    r"E:\NO-NC\normalization\revise\reviewer3_threshold_results"
)

THRESHOLDS = [5, 10, 50]
N_BOOTSTRAP = 5000
CONFIDENCE_LEVEL = 0.95
RANDOM_STATE = 20261101
N_ECE_BINS = 10

OUTPUT_CSV = RESULTS_DIR / "independent_test_metrics_bootstrap_ci.csv"


# -------------------- 指标函数 --------------------

def expected_calibration_error(y_true, score, n_bins=10):
    """
    与训练模型 notebook 一致：
    先将回归预测分数裁剪到 [0, 1]，再计算等宽分箱 ECE。
    """
    y_true = np.asarray(y_true, dtype=int)
    probability = np.clip(np.asarray(score, dtype=float), 0.0, 1.0)

    edges = np.linspace(0.0, 1.0, n_bins + 1)
    bin_id = np.minimum(
        np.digitize(probability, edges[1:-1], right=False),
        n_bins - 1
    )

    ece = 0.0
    for b in range(n_bins):
        mask = bin_id == b
        if mask.any():
            observed_rate = y_true[mask].mean()
            mean_score = probability[mask].mean()
            ece += mask.mean() * abs(observed_rate - mean_score)

    return float(ece)


def calculate_metrics(y_true, score, predicted_class):
    """按训练模型 notebook 的定义计算九项测试集指标。"""
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    predicted_class = np.asarray(predicted_class, dtype=int)

    tn = int(((y_true == 0) & (predicted_class == 0)).sum())
    fp = int(((y_true == 0) & (predicted_class == 1)).sum())

    return {
        "Accuracy": accuracy_score(y_true, predicted_class),
        "Precision": precision_score(
            y_true, predicted_class, zero_division=0
        ),
        "Balanced_accuracy": balanced_accuracy_score(
            y_true, predicted_class
        ),
        "MCC": matthews_corrcoef(y_true, predicted_class),
        "ROC_AUC": roc_auc_score(y_true, score),
        # Notebook 中的 PR-AUC 实际使用 average_precision_score
        "PR_AUC": average_precision_score(y_true, score),
        "TPR": recall_score(y_true, predicted_class, zero_division=0),
        "FPR": fp / (fp + tn) if (fp + tn) else np.nan,
        "ECE_10_equal_width_bins": expected_calibration_error(
            y_true, score, n_bins=N_ECE_BINS
        ),
    }


# -------------------- 分层 bootstrap --------------------

def stratified_bootstrap_ci(
    y_true,
    score,
    predicted_class,
    n_bootstrap=5000,
    confidence_level=0.95,
    random_state=20261101,
):
    """
    在阳性和阴性样本内分别有放回抽样，保持测试集原有类别数量。
    因此得到的区间是基于当前测试集类别构成的条件 bootstrap 区间。
    """
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)
    predicted_class = np.asarray(predicted_class, dtype=int)

    pos_idx = np.flatnonzero(y_true == 1)
    neg_idx = np.flatnonzero(y_true == 0)

    if len(pos_idx) == 0 or len(neg_idx) == 0:
        raise ValueError("测试集中必须同时有阳性和阴性样本。")

    rng = np.random.default_rng(random_state)
    metric_names = list(
        calculate_metrics(y_true, score, predicted_class).keys()
    )
    bootstrap_values = {name: [] for name in metric_names}

    for _ in range(n_bootstrap):
        # 分层抽样：每次重复保留原测试集的阳性、阴性样本数
        sampled_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True)
        sampled_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        sampled_idx = np.concatenate([sampled_pos, sampled_neg])
        rng.shuffle(sampled_idx)

        # 分层抽样保证两个类别均存在，ROC-AUC/AP 可以计算
        values = calculate_metrics(
            y_true[sampled_idx],
            score[sampled_idx],
            predicted_class[sampled_idx],
        )

        for name, value in values.items():
            bootstrap_values[name].append(value)

    alpha = 1.0 - confidence_level
    lower_q = alpha / 2
    upper_q = 1 - alpha / 2

    point_estimates = calculate_metrics(y_true, score, predicted_class)
    ci = {}

    for name in metric_names:
        arr = np.asarray(bootstrap_values[name], dtype=float)
        ci[name] = (
            float(point_estimates[name]),
            float(np.quantile(arr, lower_q)),
            float(np.quantile(arr, upper_q)),
        )

    return ci


# -------------------- 读取结果并汇总 --------------------

records = []

for threshold in THRESHOLDS:
    folder = RESULTS_DIR / f"threshold_{threshold}"
    predictions_path = folder / "independent_test_predictions.csv"
    metrics_path = folder / "independent_test_metrics.csv"
    info_path = folder / "final_model_info.json"

    if not predictions_path.exists():
        raise FileNotFoundError(f"未找到文件：{predictions_path}")

    predictions = pd.read_csv(predictions_path)
    required_cols = {
        "y_true",
        "prediction_score",
        "predicted_class",
        "classification_threshold",
    }
    missing = required_cols - set(predictions.columns)
    if missing:
        raise ValueError(
            f"{predictions_path} 缺少必要列：{sorted(missing)}"
        )

    y = predictions["y_true"].to_numpy(dtype=int)
    score = predictions["prediction_score"].to_numpy(dtype=float)
    predicted = predictions["predicted_class"].to_numpy(dtype=int)
    cutoff = float(predictions["classification_threshold"].iloc[0])

    if not np.isfinite(score).all():
        raise ValueError(f"{threshold} μg/L 的预测分数含非有限值。")
    if not set(np.unique(y)).issubset({0, 1}):
        raise ValueError(f"{threshold} μg/L 的 y_true 不是二元标签。")
    if not set(np.unique(predicted)).issubset({0, 1}):
        raise ValueError(
            f"{threshold} μg/L 的 predicted_class 不是二元标签。"
        )
    if not np.allclose(
        predictions["classification_threshold"].to_numpy(dtype=float),
        cutoff
    ):
        raise ValueError(
            f"{threshold} μg/L 的测试集记录含多个分类阈值。"
        )

    # 读取最终模型信息（若文件存在），用于结果表标注
    model_name = ""
    feature_set = ""
    if info_path.exists():
        import json
        with open(info_path, "r", encoding="utf-8") as f:
            info = json.load(f)
        model_name = info.get("model", "")
        feature_set = info.get("feature_set", "")

    ci = stratified_bootstrap_ci(
        y_true=y,
        score=score,
        predicted_class=predicted,
        n_bootstrap=N_BOOTSTRAP,
        confidence_level=CONFIDENCE_LEVEL,
        random_state=RANDOM_STATE + threshold,
    )

    n = len(y)
    n_positive = int((y == 1).sum())
    n_negative = int((y == 0).sum())

    print(
        f"{threshold} μg/L: n={n}, positive={n_positive}, "
        f"negative={n_negative}, cutoff={cutoff:.3f}"
    )

    for metric, (estimate, lower, upper) in ci.items():
        records.append({
            "target_threshold_ug_L": threshold,
            "model": model_name,
            "feature_set": feature_set,
            "classification_threshold": cutoff,
            "metric": metric,
            "estimate": estimate,
            "CI_95_lower": lower,
            "CI_95_upper": upper,
            "n_test": n,
            "n_positive": n_positive,
            "n_negative": n_negative,
            "bootstrap_replicates": N_BOOTSTRAP,
            "bootstrap_method": "stratified percentile bootstrap",
        })

results = pd.DataFrame(records)
results.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print(f"\n已保存：{OUTPUT_CSV}")
print(results.to_string(index=False))

C:\ProgramData\Anaconda3\lib\site-packages\numpy\_distributor_init.py:30: UserWarning: loaded more than 1 DLL from .libs:
C:\ProgramData\Anaconda3\lib\site-packages\numpy\.libs\libopenblas.JPIJNSWNNAN3CE6LLI5FWSPHUT2VXMTH.gfortran-win_amd64.dll
C:\ProgramData\Anaconda3\lib\site-packages\numpy\.libs\libopenblas.WCDJNK7YVMPZQ2ME2ZZHJJRJ3JIKNDB7.gfortran-win_amd64.dll
  warnings.warn("loaded more than 1 DLL from .libs:"


5 μg/L: n=808, positive=118, negative=690, cutoff=0.200
10 μg/L: n=808, positive=52, negative=756, cutoff=0.150
50 μg/L: n=808, positive=9, negative=799, cutoff=0.200

已保存：E:\NO-NC\normalization\revise\reviewer3_threshold_results\independent_test_metrics_bootstrap_ci.csv
 target_threshold_ug_L                 model feature_set  classification_threshold                  metric  estimate  CI_95_lower  CI_95_upper  n_test  n_positive  n_negative  bootstrap_replicates                bootstrap_method
                     5 RandomForestRegressor         all                      0.20                Accuracy  0.801980     0.773515     0.827970     808         118         690                  5000 stratified percentile bootstrap
                     5 RandomForestRegressor         all                      0.20               Precision  0.391753     0.341709     0.443182     808         118         690                  5000 stratified percentile bootstrap
                     5 RandomForestRegres